<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# 01 · Introducción a la automatización

## Laboratorio: un proceso que podemos explicar

**Datos sintéticos. Sin red, credenciales ni archivos reales de contribuyentes.**

El objetivo es comprender entrada, reglas, resultados, incidencias y verificación. Este bloque prepara el proyecto territorial, pero no realiza descargas, cruces con INE ni exportaciones Excel.

## Objetivos y límites

Al terminar podrás explicar una automatización, ejecutar reglas técnicas y comprobar el resultado. Trabajamos con listas de diccionarios en memoria. No se exige pandas. Los adaptadores de archivos, APIs, informes e IA pertenecen a temas posteriores.

**Requisitos:** Python 3.10+ y un entorno Jupyter para abrir este archivo. Los scripts .py solo necesitan Python. No hay que instalar paquetes para la práctica.

## Cómo trabajar

1. Ejecuta la parte guiada en orden.
2. Predice las salidas antes de ejecutar.
3. Completa los doce ejercicios independientes.
4. En cada ejercicio activa `completado_ejXX = True` para ejecutar sus controles.
5. Reinicia el kernel y ejecuta todo al terminar.

Los controles comprueban casos concretos. Las respuestas interpretativas necesitan revisión humana.

## 1. Preparación

Importamos herramientas de la biblioteca estándar. Decimal representa cantidades decimales; Counter cuenta motivos; deepcopy permite contrastar que no modificamos la entrada. re se utiliza para comprobar un formato explícito de importe.

In [ ]:
from decimal import Decimal, InvalidOperation
from collections import Counter
from copy import deepcopy
import re

ANIOS_PERMITIDOS = {2024, 2025}
VERSION_REGLAS = "aula-1.0"


## 2. El encargo

Necesitamos recibir registros sintéticos de una publicación y entregar un resumen verificable. El responsable debe poder revisar las filas excluidas.

**No estamos determinando la deuda real ni quién debería aparecer legalmente en un listado.** Aplicamos condiciones técnicas de aula. Los años permitidos son configuración fija para reproducir el resultado.

### Contrato de entrada y salida

Entrada: lista de registros con id de texto, importe de texto y año entero. El importe admite punto decimal y hasta dos decimales, sin separador de miles, positivo y con hasta doce cifras enteras.

Salida: aceptados, incidencias con motivo, decisiones por fila y total decimal. Cada fila recibe un motivo principal. Una repetición de clave válida (id, año) se considera duplicada después de validar los campos.

In [ ]:
registros = [{'id': 'DEMO-001', 'importe': '1200000.00', 'anio': 2025}, {'id': 'DEMO-002', 'importe': '800000.00', 'anio': 2025}, {'id': 'DEMO-003', 'importe': 'pendiente', 'anio': 2025}, {'id': '   ', 'importe': '700000.00', 'anio': 2025}, {'id': 'DEMO-001', 'importe': '1200000.00', 'anio': 2025}, {'id': 'DEMO-004', 'importe': '900000.00', 'anio': 2099}]


### Predicción manual

Sin ejecutar el procesador, revisa las filas. ¿Cuántos registros deberían contribuir al total? ¿Qué significa "pendiente"? ¿Podemos sumar el duplicado?

**Resultado de referencia de la parte guiada:** dos aceptados, cuatro incidencias y total aceptado de 2.000.000,00 €. Explica cómo llegarías a esta cifra.

## 3. Una operación manual que oculta un error

La siguiente suma parece funcionar, pero ignora otras reglas. Solo intenta convertir importes: incluye el duplicado, el ID vacío y un año incompatible. Capturamos únicamente la conversión esperada para mostrar el problema.

In [ ]:
total_ingenuo = Decimal("0")
for fila in registros:
    try:
        total_ingenuo += Decimal(fila["importe"])
    except InvalidOperation:
        pass  # Aquí ocultamos un problema: no dejamos evidencia.
print("Total ingenuo:", total_ingenuo)


### Diagnóstico

El resultado ingenuo es 4.800.000,00 €. Que el código termine no implica que respete el encargo. Ignorar una conversión tampoco equivale a gestionar una incidencia. Necesitamos definir el criterio completo antes de automatizarlo.

## 4. Regla de identificación

Comprobamos el tipo antes de quitar espacios. Si la entrada no es texto o está vacía, devolvemos None. No inferimos identidad, no validamos NIF y no cambiamos mayúsculas sin un criterio declarado.

In [ ]:
def normalizar_id(valor):
    """Admite texto no vacío. No valida NIF ni identidad jurídica."""
    if not isinstance(valor, str):
        return None
    limpio = valor.strip()
    return limpio if limpio else None


In [ ]:
print(normalizar_id(" DEMO-001 "))
print(normalizar_id("   "))
print(normalizar_id(None))
assert normalizar_id(" DEMO-001 ") == "DEMO-001"
assert normalizar_id(None) is None


## 5. Regla de importe

Primero comprobamos el formato acordado y después construimos Decimal desde texto. Rechazar valores ambiguos hace visible el problema. No rellenamos con cero ni adivinamos si una coma es separador de miles o decimal.

En un tema posterior añadiremos normalización declarada para formatos de entrada concretos.

In [ ]:
def convertir_importe(valor):
    """Contrato: texto positivo, punto decimal y hasta 2 decimales.

    Como máximo 12 cifras enteras. No se adivina el formato local.
    Un dato no interpretable devuelve None, nunca cero.
    """
    if not isinstance(valor, str):
        return None
    texto = valor.strip()
    if not re.fullmatch(r"[0-9]{1,12}(?:\.[0-9]{1,2})?", texto):
        return None
    try:
        importe = Decimal(texto)
    except InvalidOperation:
        return None
    if not importe.is_finite() or importe <= 0:
        return None
    return importe.quantize(Decimal("0.01"))


In [ ]:
for muestra in ["1200.50", "1.200,50", "NaN", "Infinity", "-3", "0", "1.234", "25"]:
    print(repr(muestra), "=>", convertir_importe(muestra))
assert convertir_importe("1200.50") == Decimal("1200.50")
assert convertir_importe("NaN") is None


### Qué aporta Decimal y qué no aporta

Evita introducir la aproximación binaria de un float antes de construir el importe. No decide si la fuente es fiable ni si una deuda está vigente. El contexto de precisión sigue existiendo; nuestro contrato acota las cifras de los ejemplos.

Documentación: https://docs.python.org/3/library/decimal.html

## 6. Regla de año y clave compuesta

Los años permitidos se declaran por ejecución. No dependen de la fecha del ordenador. Excluimos bool aunque Python lo relacione con int. La clave de duplicado incluye id y año, porque una entidad puede figurar en dos ediciones.

In [ ]:
def anio_valido(valor, anios_permitidos):
    # bool es una subclase de int: el contrato lo excluye expresamente.
    return type(valor) is int and valor in anios_permitidos


In [ ]:
print(anio_valido(2025, ANIOS_PERMITIDOS))
print(anio_valido("2025", ANIOS_PERMITIDOS))
print(anio_valido(True, ANIOS_PERMITIDOS))
assert anio_valido(2025, ANIOS_PERMITIDOS)
assert not anio_valido(True, ANIOS_PERMITIDOS)


## 7. Una decisión por registro

Orden: estructura, ID, importe, año, duplicado y aceptación. Una fila con varios problemas obtiene el primero. Un registro inválido no ocupa la clave. Conservamos el primer registro técnicamente válido por clave en este ejercicio.

**Límite:** dos importes diferentes para una clave real requerirían revisión y una política adicional, no una selección silenciosa.

In [ ]:
def validar_fila(fila, vistos, anios_permitidos):
    """Aplica el primer motivo de rechazo; vistos cambia solo al aceptar.

    Se admiten diccionarios; una fila de otro tipo es una incidencia.
    No modifica el diccionario original.
    """
    if not isinstance(fila, dict):
        return {"estado": "estructura_invalida", "motivo": "La fila no es un diccionario"}
    identificador = normalizar_id(fila.get("id"))
    if identificador is None:
        return {"estado": "id_vacio", "motivo": "ID ausente o de tipo no admitido"}
    importe = convertir_importe(fila.get("importe"))
    if importe is None:
        return {"estado": "importe_invalido", "motivo": "Importe fuera del contrato"}
    anio = fila.get("anio")
    if not anio_valido(anio, anios_permitidos):
        return {"estado": "anio_invalido", "motivo": "Año fuera del contrato"}
    clave = (identificador, anio)
    if clave in vistos:
        return {"estado": "duplicado", "motivo": "Clave válida repetida en esta ejecución"}
    vistos.add(clave)
    return {"id": identificador, "importe": importe, "anio": anio,
            "estado": "aceptado", "motivo": "Supera los controles técnicos"}


In [ ]:
vistos_demo = set()
print(validar_fila(registros[0], vistos_demo, ANIOS_PERMITIDOS))
print(validar_fila(registros[0], vistos_demo, ANIOS_PERMITIDOS))


## 8. El proceso completo

El conjunto de claves nace dentro de cada llamada para que una ejecución no contamine la siguiente. Las incidencias conservan una copia de la entrada y el número de fila. Una entrada global incompatible detiene el proceso. Una fila inválida se registra sin impedir revisar las demás.

In [ ]:
def procesar_registros(registros, anios_permitidos=None):
    """Devuelve resultado funcional determinista y evidencia por fila.

    Política docente: para una clave se conserva el primer registro válido.
    Un conflicto real requeriría una política de resolución adicional.
    """
    if not isinstance(registros, list):
        raise TypeError("La entrada debe ser una lista de registros")
    if anios_permitidos is None:
        anios_permitidos = ANIOS_PERMITIDOS
    if not isinstance(anios_permitidos, (set, frozenset)) or not all(
        type(anio) is int for anio in anios_permitidos
    ):
        raise TypeError("Los años permitidos deben ser un conjunto de enteros")
    vistos = set()
    aceptados, incidencias, decisiones = [], [], []
    for numero, fila in enumerate(registros, start=1):
        resultado = validar_fila(fila, vistos, anios_permitidos)
        evidencia = {"fila": numero, **resultado}
        decisiones.append(evidencia)
        if resultado["estado"] == "aceptado":
            aceptados.append(evidencia)
        else:
            incidencias.append({**evidencia, "original": deepcopy(fila)})
    if len(registros) != len(aceptados) + len(incidencias):
        raise RuntimeError("La reconciliación de filas ha fallado")
    total = sum((f["importe"] for f in aceptados), start=Decimal("0.00"))
    return {"version_reglas": VERSION_REGLAS,
            "anios_permitidos": sorted(anios_permitidos),
            "n_entrada": len(registros), "aceptados": aceptados,
            "incidencias": incidencias, "decisiones": decisiones,
            "total_aceptado": total}


In [ ]:
resultado = procesar_registros(registros)
for decision in resultado["decisiones"]:
    print(decision["fila"], decision["estado"], decision["motivo"])


## 9. Resumen e indicadores

La suma solo incluye aceptados. La tasa de incidencias divide por filas de entrada y devuelve None sin denominador. No es una medida del error oculto del sistema: algunos fallos pueden no detectarse con estas reglas.

In [ ]:
def porcentaje(parte, total):
    if total == 0:
        return None
    return 100 * parte / total


def resumir(resultado):
    return {
        "entrada": resultado["n_entrada"],
        "aceptados": len(resultado["aceptados"]),
        "incidencias": len(resultado["incidencias"]),
        "total_aceptado": resultado["total_aceptado"],
        "porcentaje_incidencias": porcentaje(
            len(resultado["incidencias"]), resultado["n_entrada"]
        ),
        "motivos": dict(Counter(f["estado"] for f in resultado["incidencias"])),
    }


In [ ]:
resumen = resumir(resultado)
for clave, valor in resumen.items():
    print(clave, ":", valor)
assert resumen["aceptados"] == 2
assert resumen["incidencias"] == 4
assert resumen["total_aceptado"] == Decimal("2000000.00")


### Representación de los estados sin librerías externas

Cada bloque representa un registro. El objetivo es interpretar la distribución; los gráficos editoriales están en la presentación.

In [ ]:
for estado, cantidad in Counter(f["estado"] for f in resultado["decisiones"]).items():
    print(f"{estado:20} {'█' * cantidad} ({cantidad})")


## 10. Comprobaciones con propósito

Comprobamos conservación de filas, un total conocido, repetición y ausencia de mutación. Las aserciones ayudan en laboratorio. En una ejecución operativa, las validaciones que deban actuar siempre se implementan con condiciones explícitas, como la reconciliación del procesador.

In [ ]:
entrada_antes = deepcopy(registros)
primera = procesar_registros(registros)
segunda = procesar_registros(registros)
assert primera == segunda
assert registros == entrada_antes
assert primera["n_entrada"] == len(primera["aceptados"]) + len(primera["incidencias"])
print("Resultado repetible y entrada intacta")


### Lista vacía y estructura incompatible

Una lista vacía puede ser una entrada válida sin observaciones. No demuestra que una descarga previa haya funcionado. Una cadena en lugar de lista incumple el contrato global y provoca una excepción clara.

In [ ]:
print(resumir(procesar_registros([])))
try:
    procesar_registros("esto no es una lista")
except TypeError as error:
    print("Error global esperado:", error)


## 11. Interpretación y supervisión

Podemos decir que dos registros sintéticos superan controles técnicos. No podemos inferir municipio, evolución real ni pago. La revisión humana debe comprobar casos en conflicto y modificar reglas cuando cambie el contrato de la fuente.

La publicación original de información personal no se asume como permiso general para reutilizarla o difundirla. El trabajo de aula no usa datos reales.

## 12. Coste ilustrativo

Con $T_m(n)=n t_m$ y $T_a(n)=T_0+n t_a$, el ahorro requiere $n>T_0/(t_m-t_a)$ cuando $t_m>t_a$. Usamos minutos en todos los parámetros. El modelo excluye mantenimiento variable y cambios de formato.

In [ ]:
t0, tm, ta = Decimal("240"), Decimal("2"), Decimal("0.1")
n_equilibrio = t0 / (tm - ta)
print("Equilibrio teórico:", n_equilibrio)
print("Con 127 registros:", 127 * tm - (t0 + 127 * ta), "minutos ahorrados")


## Ejercicios

Los doce ejercicios usan únicamente la parte guiada. No necesitas resolver uno para comenzar el siguiente. Cada tarea incluye una pista y controles activables. En la versión del alumnado los controles se omiten hasta cambiar la bandera a True. No confundas «omitido» con «superado».

## Ejercicio 01 · Entrada, regla, salida y revisión

Describe el proceso en un diccionario `contrato`. Utiliza las claves entrada, regla, salida y revision. Diferencia un control técnico de una decisión sobre el significado tributario.

**Pista:** La entrada no es «un informe». Piensa qué recibe Python antes de calcular.

In [ ]:
contrato = {"entrada": "", "regla": "", "salida": "", "revision": ""}

In [ ]:
completado_ej01 = False
if completado_ej01:
    assert set(contrato) == {"entrada", "regla", "salida", "revision"}
    assert all(isinstance(v, str) and v.strip() for v in contrato.values())
    print("Ejercicio 01: controles superados")
else:
    print("Ejercicio 01: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 02 · Predicción antes de ejecutar

Escribe `esperado` con las claves aceptados, incidencias y total. Predice el resultado de las seis filas sin llamar al procesador. El total debe ser Decimal.

**Pista:** El duplicado no vuelve a contribuir y los errores no se imputan como cero.

In [ ]:
esperado = {"aceptados": None, "incidencias": None, "total": None}

In [ ]:
completado_ej02 = False
if completado_ej02:
    assert esperado == {"aceptados": 2, "incidencias": 4, "total": Decimal("2000000.00")}
    print("Ejercicio 02: controles superados")
else:
    print("Ejercicio 02: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 03 · Una regla para identificadores

Implementa `id_correcto(valor)`: True si es una cadena no vacía tras quitar espacios. Rechaza None, números y cadenas de espacios.

**Pista:** Comprueba el tipo antes de utilizar strip.

In [ ]:
def id_correcto(valor):
    # TODO: devuelve un booleano
    return None

In [ ]:
completado_ej03 = False
if completado_ej03:
    assert id_correcto(" DEMO-001 ") is True
    assert id_correcto(" ") is False
    assert id_correcto(None) is False
    assert id_correcto(123) is False
    print("Ejercicio 03: controles superados")
else:
    print("Ejercicio 03: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 04 · Casos de importe que requieren revisión

Construye `clasificacion_importes`, una lista de booleanos para ["1200.50", "1.200,50", "NaN", "0", "-2", "12.345"]. Usa el conversor y explica por qué el formato local requiere una regla adicional.

**Pista:** None representa un importe rechazado por el contrato.

In [ ]:
muestras_importe = ["1200.50", "1.200,50", "NaN", "0", "-2", "12.345"]
clasificacion_importes = []  # TODO

In [ ]:
completado_ej04 = False
if completado_ej04:
    assert clasificacion_importes == [True, False, False, False, False, False]
    print("Ejercicio 04: controles superados")
else:
    print("Ejercicio 04: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 05 · La clave incluye el año

Crea `resultado_periodos` procesando dos filas del mismo ID, importes 100 y 200, años 2024 y 2025. ¿Hay un duplicado?

**Pista:** Ambos años están permitidos y las claves compuestas son diferentes.

In [ ]:
filas_periodos = []  # TODO: dos diccionarios
resultado_periodos = None  # TODO

In [ ]:
completado_ej05 = False
if completado_ej05:
    assert len(resultado_periodos["aceptados"]) == 2
    assert not resultado_periodos["incidencias"]
    assert resultado_periodos["total_aceptado"] == Decimal("300.00")
    print("Ejercicio 05: controles superados")
else:
    print("Ejercicio 05: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 06 · Incidencias que se pueden explicar

Procesa los datos originales y crea `motivos_ej6` como un diccionario de frecuencias por estado de incidencia. Cada incidencia debe conservar el número de fila y la entrada original.

**Pista:** Counter acepta una secuencia de etiquetas.

In [ ]:
resultado_ej6 = procesar_registros(registros)
motivos_ej6 = {}  # TODO

In [ ]:
completado_ej06 = False
if completado_ej06:
    assert motivos_ej6 == {"importe_invalido":1, "id_vacio":1, "duplicado":1, "anio_invalido":1}
    assert all("original" in f and "fila" in f for f in resultado_ej6["incidencias"])
    print("Ejercicio 06: controles superados")
else:
    print("Ejercicio 06: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 07 · Un control que detecta filas perdidas

Implementa `reconciliar(entrada, aceptados, incidencias)` con los tres números de filas. Devuelve un booleano. Comprueba un caso correcto y otro con una fila perdida.

**Pista:** Se comprueba conservación de filas, no importes imposibles de interpretar.

In [ ]:
def reconciliar(entrada, aceptados, incidencias):
    return None  # TODO

In [ ]:
completado_ej07 = False
if completado_ej07:
    assert reconciliar(6, 2, 4) is True
    assert reconciliar(6, 2, 3) is False
    print("Ejercicio 07: controles superados")
else:
    print("Ejercicio 07: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 08 · Porcentajes y denominador vacío

Implementa `tasa_incidencias(n_inc, n_total)`: porcentaje o None cuando total es cero. Calcula 4/6 y explica por qué no es una estimación del error real no detectado.

**Pista:** Devuelve un número entre 0 y 100 en entradas válidas. No redondees hasta mostrar.

In [ ]:
def tasa_incidencias(n_inc, n_total):
    return None  # TODO

In [ ]:
completado_ej08 = False
if completado_ej08:
    assert abs(tasa_incidencias(4, 6) - 66.6666666667) < 1e-6
    assert tasa_incidencias(0, 0) is None
    print("Ejercicio 08: controles superados")
else:
    print("Ejercicio 08: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 09 · Repetibilidad y entrada intacta

Guarda una copia de los datos, procesa dos veces y define `repetible` y `sin_mutacion`. Ambos deben ser True. Explica por qué eso no garantiza que un envío de correo no se duplique.

**Pista:** Usa deepcopy y compara los resultados completos de este proceso sin tiempo dinámico.

In [ ]:
antes_ej9 = deepcopy(registros)
r1_ej9 = procesar_registros(registros)
r2_ej9 = procesar_registros(registros)
repetible = None  # TODO
sin_mutacion = None  # TODO

In [ ]:
completado_ej09 = False
if completado_ej09:
    assert repetible is True
    assert sin_mutacion is True
    print("Ejercicio 09: controles superados")
else:
    print("Ejercicio 09: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 10 · Casos límite del proceso

Define `resumen_vacio` procesando [] y `resumen_invalidos` procesando una fila con ID vacío e importe "mal". Usa resumir. ¿Qué motivo principal gana?

**Pista:** ID vacío se comprueba antes que el importe.

In [ ]:
resumen_vacio = None  # TODO
resumen_invalidos = None  # TODO

In [ ]:
completado_ej10 = False
if completado_ej10:
    assert resumen_vacio["total_aceptado"] == Decimal("0.00")
    assert resumen_vacio["porcentaje_incidencias"] is None
    assert resumen_invalidos["motivos"] == {"id_vacio":1}
    print("Ejercicio 10: controles superados")
else:
    print("Ejercicio 10: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 11 · El coste inicial y el punto de equilibrio

Implementa `umbral_equilibrio(t0, tm, ta)` con minutos y devuelve el mínimo número entero de registros para tener un ahorro estrictamente positivo. Devuelve None si tm <= ta.

**Pista:** Resuelve n > t0/(tm-ta). Si el cociente es entero, ese punto solo iguala el coste.

In [ ]:
def umbral_equilibrio(t0, tm, ta):
    return None  # TODO

In [ ]:
completado_ej11 = False
if completado_ej11:
    assert umbral_equilibrio(240, 2, 0.1) == 127
    assert umbral_equilibrio(10, 2, 1) == 11
    assert umbral_equilibrio(240, 1, 1) is None
    print("Ejercicio 11: controles superados")
else:
    print("Ejercicio 11: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Ejercicio 12 · Una conclusión apoyada en evidencia

Escribe `conclusion` con dos o tres frases: número de aceptados, total y necesidad de revisión. Añade `limites` con al menos dos inferencias que los datos no permiten.

**Pista:** No hay municipio, prueba de pago ni serie temporal en la entrada.

In [ ]:
conclusion = ""  # TODO
limites = []  # TODO

In [ ]:
completado_ej12 = False
if completado_ej12:
    assert isinstance(conclusion, str) and len(conclusion) > 40
    assert len(limites) >= 2 and all(isinstance(x,str) and x.strip() for x in limites)
    print("Ejercicio 12: controles superados")
else:
    print("Ejercicio 12: controles omitidos; completa la tarea y activa la bandera")

**Explica tu decisión:** añade aquí el razonamiento y qué evidencia la respalda.

## Entrega y autoevaluación

Reinicia el kernel y ejecuta todas las celdas. Entrega las doce tareas con controles activados y una explicación breve. Revisa el contrato, la conservación de filas y los límites de interpretación.

Rúbrica: especificación, ejecución, validación, trazabilidad e interpretación, 2 puntos por dimensión.

## Fuentes

- https://docs.python.org/3/tutorial/
- https://docs.python.org/3/library/decimal.html
- https://www.ine.es/datosabiertos/
- https://www.boe.es/buscar/act.php?id=BOE-A-2003-23186#a95bis
- https://www.boe.es/doue/2016/119/L00001-00088.pdf

Datos y reglas de aula ficticios. Referencias revisadas el 02/11/2026.